# Research Simulation: Does a Calm Period VaR Survive a Crisis

**Family:** End to End Banking Projects | **Level:** Applied | **Dataset:** `wti_spot_daily and fx_daily_2000` | **Data kind:** empirical | **Module status:** Validating

Hypotheses, a written decision rule and real price history from 1986, to test whether a 500 day VaR fails when it matters.

> Data are empirical.

## Table of Contents

1. [Problem Statement](#1-problem-statement)
2. [Learning Objectives](#2-learning-objectives)
3. [Fundamental Concepts](#3-fundamental-concepts)
4. [Why This Method?](#4-why-this-method)
5. [Research Foundation](#5-research-foundation)
6. [Mathematical Derivation](#6-mathematical-derivation)
7. [Worked Numerical Example](#7-worked-numerical-example)
8. [Dataset Exploration](#8-dataset-exploration)
9. [Data Preparation](#9-data-preparation)
10. [Implementation from Scratch](#10-implementation-from-scratch)
11. [Implementation using Libraries](#11-implementation-using-libraries)
12. [Model Evaluation](#12-model-evaluation)
13. [Visualization](#13-visualization)
14. [Interpretation](#14-interpretation)
15. [Experiments](#15-experiments)
16. [Results and Comparisons](#16-results-and-comparisons)
17. [Limitations and Model Risk](#17-limitations-and-model-risk)
18. [Practical Banking Application](#18-practical-banking-application)
19. [Exercises](#19-exercises)
20. [References](#20-references)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from risk_lab.reporting.results import find_root, save_config, save_result
from risk_lab.visualization.style import COLORS, apply_style, save_figure

apply_style()
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)
MODULE = "var_crisis_research"
ROOT = find_root()
SEED = 42
from scipy.stats import binomtest, t

from risk_lab.market.backtest import christoffersen_independence, kupiec_pof
from risk_lab.market.var import ewma_var, rolling_historical_var

ALPHA = 0.99
EXPECTED_RATE = 1 - ALPHA

## 1. Problem Statement

A risk committee looks at a backtest of the bank's 99 percent one day VaR. The model passed in the calm years. A member asks the
question that matters for capital: **will the same model hold in a crisis, when the loss is real?**

This notebook is a **research simulation**. It treats the question as a testable claim, fixes the rules for judging it, and runs
it on **real public price history**: daily WTI crude oil spot prices from 1986 and an equal weight basket of six currencies
against the US dollar from 2000.

Hypotheses, stated before the final run:

* **H1.** A 500 day historical VaR at 99 percent has a pooled exception rate above 1 percent in crisis windows, and the Kupiec test
  rejects the 1 percent rate at the 5 percent level, on both assets.
* **H2.** A volatility responsive model (EWMA volatility with a Student t multiplier, 4 degrees of freedom) has a pooled crisis
  exception rate closer to 1 percent than the historical model, on both assets.

A third quantity is reported without a hypothesis: **what the responsive model costs in calm periods**, measured as the ratio of
its average VaR to the historical VaR.

**Honesty note.** The author ran a quick exploratory version of this comparison before writing the rules down. The rules below were
fixed before the final run and are the same ones in the code, but this is not a blind pre registration.

## 2. Learning Objectives

After this notebook you can:

1. Turn a vague worry ("will the model hold in a crisis?") into hypotheses with a written decision rule.
2. Build returns from real price data and document every day that had to be excluded and why.
3. Implement a rolling historical VaR and an EWMA VaR from scratch and match the library with no look ahead.
4. Compare models on the same days, by period, with Kupiec tests and exact binomial intervals.
5. Test whether the conclusion survives changes in the window length, the decay and the degrees of freedom.
6. State the cost of a more responsive model, not only its benefit.
7. Write the finding in the form a risk committee can act on, with its limits.

## 3. Fundamental Concepts

| Term | Meaning |
|---|---|
| Crisis window | A calendar interval fixed in advance from well known market events. It is not chosen by how the model performed |
| Pooled exception rate | Exceptions divided by days, with all days of all crisis windows put together |
| Calm period | Every evaluated day outside the crisis windows |
| Exception | A day when the loss was larger than the one day VaR forecast made the day before |
| Kupiec test | Tests whether the exception rate equals the expected 1 percent |
| Exact interval | Clopper Pearson interval for a binomial rate. Too narrow when exceptions cluster |
| EWMA | Exponentially weighted moving average of squared returns, decay 0.94 here |
| Procyclicality | A risk measure that is low before a crisis and jumps in it, so it asks for capital when capital is scarce |

## 4. Why This Method?

**Why a research simulation on real data?** The earlier modules showed that VaR code is correct on synthetic data. They could not
show whether a method fails when the world changes, because the generator never changed. Real history contains regime breaks the author did
not design.

**Why these two models?** The historical 500 day window is a common baseline. EWMA with a fat tailed multiplier is the simplest
model that reacts to a volatility jump. Comparing a slow model with a fast model is the cleanest test of the claim.

**Method checks (acceptance criteria).** These check the method, not the finding. A finding can go either way and is still valid.

1. The from scratch VaR for five random days equals the library to 1e-12 using only data before the day (no look ahead).
2. Every crisis window lies inside the data and holds at least 80 evaluated days for each asset.
3. The count of excluded WTI return days equals the count found by an independent rule.
4. The two hypotheses are evaluated with the stated rules for both assets.

## 5. Research Foundation

### Academic and book sources

- **Techniques for verifying the accuracy of risk measurement models** (Kupiec, P. H., 1995). Introduces the proportion of failures likelihood ratio test for VaR exceptions and discusses its low power on short samples. The Journal of Derivatives version was not checked. [verified: Fed FRASER and EconPapers records, web search 2026-11-25]
- **Evaluating interval forecasts** (Christoffersen, P. F., 1998). Defines unconditional coverage, independence and conditional coverage tests for interval forecasts. VaR exceptions are a special case. [verified: RePEc IDEAS and EconPapers records, web search 2026-11-25]

### Regulatory and standard setter sources

- **Minimum capital requirements for market risk (BCBS d457)** (Basel Committee on Banking Supervision, 2019). Status: integrated into the consolidated Basel Framework. Revised market risk framework. The internal models approach uses expected shortfall, and the standard supersedes the January 2016 version (d352). [verified: BIS page, web search 2026-11-25]

**How this notebook uses them.** Kupiec (1995) and Christoffersen (1998) supply the tests. The Basel market risk standard is the supervisory setting in which exception counts matter. The module does not reproduce a published experiment. It poses its own question on public data and reports the answer with its limits.

**Academic findings versus regulatory requirements.** Papers describe what methods can do. Regulations and accounting standards say what a supervised institution must do. This notebook labels which is which, and it does not reproduce any paper's experiment. Data differ, so numbers are not comparable with published results.

## 6. Mathematical Derivation

**Return.** With price $P_t$, the simple return is $r_t = P_t / P_{t-1} - 1$ and the loss is $\ell_t = -r_t$. For a currency quoted as units
per US dollar, $S_t$, the dollar value of one unit is $1/S_t$, so the dollar return of holding it is $r_t = S_{t-1}/S_t - 1$.

**Historical VaR.** With window $w$, $\widehat{\mathrm{VaR}}_t = $ empirical $\alpha$ quantile of $\ell_{t-w}, \dots, \ell_{t-1}$.

**EWMA VaR.** $\sigma_t^2 = \lambda \sigma_{t-1}^2 + (1-\lambda) r_{t-1}^2$ and

$$\widehat{\mathrm{VaR}}_t = \sigma_t \, t_\alpha(\nu)\sqrt{(\nu-2)/\nu}$$

with $t_\alpha(\nu)$ the Student t quantile. The factor $\sqrt{(\nu-2)/\nu}$ gives the t distribution unit variance.

**Test.** Over $T$ days with $x$ exceptions, the Kupiec statistic is
$LR = -2\ln[(1-p)^{T-x}p^x] + 2\ln[(1-x/T)^{T-x}(x/T)^x]$, $p = 0.01$, chi squared with one degree of freedom.

**Why 500 days is thin.** At 99 percent, a 500 day window has $500 \times 0.01 = 5$ days beyond the VaR. After a long calm stretch the
five worst days are mild, so the VaR is low just when volatility is about to rise.

## 7. Worked Numerical Example

A tiny example with a 10 day window and 80 percent VaR so that the numbers can be checked by eye. Ten past losses
(percent): 0.2, -0.4, 0.5, 0.1, -0.3, 0.9, -0.1, 0.3, 0.6, -0.5. Sorted: -0.5, -0.4, -0.3, -0.1, 0.1, 0.2, 0.3, 0.5, 0.6, 0.9.
With linear interpolation the 80 percent quantile sits at position $9 \times 0.8 = 7.2$ from zero, between 0.5 and 0.6.

In [2]:
past_losses = np.array([0.2, -0.4, 0.5, 0.1, -0.3, 0.9, -0.1, 0.3, 0.6, -0.5])
ordered = np.sort(past_losses)
position = (len(ordered) - 1) * 0.8
low = int(np.floor(position))
by_hand = ordered[low] + (position - low) * (ordered[low + 1] - ordered[low])
print("position:", round(position, 2), "VaR by hand:", round(by_hand, 4))
print("numpy quantile:", round(float(np.quantile(past_losses, 0.8)), 4))
assert abs(by_hand - np.quantile(past_losses, 0.8)) < 1e-12
# exception rule
today_loss = 0.7
print("today loss", today_loss, "exception:", today_loss > by_hand)

position: 7.2 VaR by hand: 0.52
numpy quantile: 0.52
today loss 0.7 exception: True


Today's loss of 0.7 is larger than the VaR of 0.52, so it counts as one exception. The rest of the notebook counts such days on
thousands of real days and asks whether the count is close to 1 in 100.

## 8. Dataset Exploration

In [3]:
wti_raw = pd.read_csv(ROOT / "datasets" / "raw" / "wti_spot_daily.csv", parse_dates=["Date"]).set_index("Date")["Price"]
fx_raw = pd.read_csv(ROOT / "datasets" / "processed" / "fx_daily_2000.csv", parse_dates=["Date"]).set_index("Date")
print("WTI days:", len(wti_raw), "from", wti_raw.index.min().date(), "to", wti_raw.index.max().date())
print("WTI price range:", wti_raw.min(), "to", wti_raw.max())
print()
print("WTI days with price at or below zero:")
print(wti_raw[wti_raw <= 0])
print()
print("FX days:", len(fx_raw), "from", fx_raw.index.min().date(), "to", fx_raw.index.max().date())
print("share of missing days per currency:")
print(fx_raw.isna().mean().round(3).to_string())

WTI days: 9547 from 1986-01-02 to 2026-11-17
WTI price range: -36.98 to 145.31

WTI days with price at or below zero:
Date
2020-04-20   -36.98
Name: Price, dtype: float64

FX days: 7015 from 2000-01-03 to 2026-11-20
share of missing days per currency:
Canada            0.039
Euro              0.039
India             0.039
Japan             0.039
Malaysia          0.039
Singapore         0.039
South Korea       0.039
Switzerland       0.039
Thailand          0.039
United Kingdom    0.039


Two facts from the real data matter before any model is fitted. First, the WTI series has one day with a **negative price**, so a
return from that day is not defined. Second, the currency file has holiday gaps, so some days are missing for all currencies at once.
Both are handled openly in the next section.

## 9. Data Preparation

In [4]:
# wti returns
has_previous = wti_raw.shift().notna()
valid = (wti_raw > 0) & (wti_raw.shift() > 0)
excluded_days = wti_raw.index[has_previous & ~valid]
wti_ret = wti_raw.pct_change()[valid]
print("excluded WTI return days:", [d.date().isoformat() for d in excluded_days])
print("WTI returns:", len(wti_ret))

# independent exclusion count
nonpositive = wti_raw.index[wti_raw <= 0]
expected_excluded = set(nonpositive) | {wti_raw.index[wti_raw.index.get_loc(d) + 1] for d in nonpositive}
print("independent rule gives:", len(expected_excluded), "days")

# fx basket
currencies = ["Euro", "Japan", "United Kingdom", "Switzerland", "Canada", "Singapore"]
fx = fx_raw[currencies].dropna()
assert fx["Euro"].mean() < 1 and fx["United Kingdom"].mean() < 1, "quote is not per US dollar"
fx_each = fx.shift() / fx - 1
fx_each = fx_each.dropna()
basket_ret = fx_each.mean(axis=1)
print("FX basket returns:", len(basket_ret), "days dropped for gaps:", len(fx_raw) - len(fx))

assets = {"WTI": wti_ret, "FX basket": basket_ret}
windows = {
    "gfc": ("2008-09-01", "2009-06-30"),
    "oil_collapse": ("2014-06-01", "2016-02-29"),
    "covid": ("2020-02-20", "2020-06-30"),
    "ukraine": ("2022-02-24", "2022-12-31"),
}

excluded WTI return days: ['2020-04-20', '2020-04-21']
WTI returns: 9544
independent rule gives: 2 days
FX basket returns: 6740 days dropped for gaps: 274


**Decisions recorded.** (1) Days with an undefined return are dropped, not repaired. Repairing would invent a number for the most extreme
day in the file. (2) Missing currency days are dropped for all currencies, so a return can span a holiday. (3) The basket holds equal dollar amounts of six currencies,
rebalanced daily. The Indonesian rupiah is not in the source. (4) The crisis windows are fixed calendar dates taken from well known
events. The dates are the author's judgment, not from a verified source.

## 10. Implementation from Scratch

In [5]:
def hist_var_scratch(x, i, window, alpha):
    # sort past losses
    past = np.sort(-x[i - window:i])
    position = (window - 1) * alpha
    low = int(np.floor(position))
    return past[low] + (position - low) * (past[low + 1] - past[low])


def ewma_var_scratch(x, i, decay, nu, alpha):
    # variance recursion
    variance = x[:30].var()
    for k in range(1, i + 1):
        variance = decay * variance + (1 - decay) * x[k - 1] ** 2
    multiplier = t.ppf(alpha, nu) * np.sqrt((nu - 2) / nu)
    return np.sqrt(variance) * multiplier


# library series
def model_series(r):
    x = r.to_numpy()
    raw = {
        "hist250": rolling_historical_var(x, 250, ALPHA),
        "hist500": rolling_historical_var(x, 500, ALPHA),
        "hist1000": rolling_historical_var(x, 1000, ALPHA),
        "ewma_normal": ewma_var(x, 0.94, ALPHA),
        "ewma_t4": ewma_var(x, 0.94, ALPHA, nu=4),
    }
    return pd.DataFrame(raw, index=r.index)


var_tables = {name: model_series(r) for name, r in assets.items()}

rng = np.random.default_rng(SEED)
worst_gap = 0.0
for name, r in assets.items():
    x = r.to_numpy()
    for i in rng.integers(1500, len(x), size=5):
        gap_h = abs(hist_var_scratch(x, i, 500, ALPHA) - var_tables[name]["hist500"].iloc[i])
        gap_e = abs(ewma_var_scratch(x, i, 0.94, 4, ALPHA) - var_tables[name]["ewma_t4"].iloc[i])
        worst_gap = max(worst_gap, gap_h, gap_e)
print("largest gap, scratch against library:", f"{worst_gap:.2e}")
assert worst_gap < 1e-12

largest gap, scratch against library: 0.00e+00


The scratch version of each VaR uses only data strictly before day $i$ (`x[i-window:i]` and a recursion that stops at $x_{i-1}$). It
matches the library for five random days per asset, so the series are free of look ahead.

## 11. Implementation using Libraries

In [6]:
def exception_series(r, var_series):
    # one if loss above VaR
    valid = var_series.notna()
    return pd.Series((-r[valid] > var_series[valid]).astype(int), index=r.index[valid])


def crisis_flag(index):
    flag = pd.Series(False, index=index)
    for start, end in windows.values():
        flag[start:end] = True
    return flag


# common evaluation days
hits = {}
evaluation_days = {}
for name, r in assets.items():
    table = var_tables[name]
    common = table.dropna().index
    evaluation_days[name] = common
    hits[name] = {model: exception_series(r.loc[common], table.loc[common, model]) for model in table.columns}
    print(name, "evaluation days:", len(common), common.min().date(), "to", common.max().date())

WTI evaluation days: 8544 1990-02-26 to 2026-11-17
FX basket evaluation days: 5740 2003-12-29 to 2026-11-20


All five models are scored on the **same days**, those after the longest window (1000 days) has filled. Scoring models on different days
would hide differences in the sample, not in the model.

## 12. Model Evaluation

In [7]:
def rate_row(asset, model, period, h):
    n = int(len(h))
    k = int(h.sum())
    kup = kupiec_pof(h.to_numpy(), EXPECTED_RATE)
    low, high = binomtest(k, n, EXPECTED_RATE).proportion_ci(method="exact")
    return {"asset": asset, "model": model, "period": period, "days": n, "exceptions": k,
            "rate": k / n, "kupiec_p": kup["p_value"], "ci_low": low, "ci_high": high}


rows = []
for name in assets:
    flag = crisis_flag(hits[name]["hist500"].index)
    for model, h in hits[name].items():
        for label, (start, end) in windows.items():
            rows.append(rate_row(name, model, label, h[start:end]))
        rows.append(rate_row(name, model, "crisis_pooled", h[flag]))
        rows.append(rate_row(name, model, "calm", h[~flag]))
evaluation = pd.DataFrame(rows)
view = evaluation[evaluation["period"].isin(["crisis_pooled", "calm"])]
print(view.round(4).to_string(index=False))

    asset       model        period  days  exceptions   rate  kupiec_p  ci_low  ci_high
      WTI     hist250 crisis_pooled   941          33 0.0351    0.0000  0.0243   0.0489
      WTI     hist250          calm  7603         105 0.0138    0.0016  0.0113   0.0167
      WTI     hist500 crisis_pooled   941          50 0.0531    0.0000  0.0397   0.0695
      WTI     hist500          calm  7603          83 0.0109    0.4286  0.0087   0.0135
      WTI    hist1000 crisis_pooled   941          54 0.0574    0.0000  0.0434   0.0742
      WTI    hist1000          calm  7603          70 0.0092    0.4812  0.0072   0.0116
      WTI ewma_normal crisis_pooled   941          22 0.0234    0.0004  0.0147   0.0352
      WTI ewma_normal          calm  7603         145 0.0191    0.0000  0.0161   0.0224
      WTI     ewma_t4 crisis_pooled   941          12 0.0128    0.4157  0.0066   0.0222
      WTI     ewma_t4          calm  7603          86 0.0113    0.2603  0.0091   0.0140
FX basket     hist250 crisis_poo

In [8]:
# hypothesis rules
outcomes = {}
for name in assets:
    sub = evaluation[(evaluation["asset"] == name) & (evaluation["period"] == "crisis_pooled")].set_index("model")
    h1 = bool(sub.loc["hist500", "rate"] > EXPECTED_RATE and sub.loc["hist500", "kupiec_p"] < 0.05)
    gap_hist = abs(sub.loc["hist500", "rate"] - EXPECTED_RATE)
    gap_ewma = abs(sub.loc["ewma_t4", "rate"] - EXPECTED_RATE)
    outcomes[name] = {"H1": h1, "H2": bool(gap_ewma < gap_hist),
                      "gap_hist500": float(gap_hist), "gap_ewma_t4": float(gap_ewma)}
overall = {"H1": all(v["H1"] for v in outcomes.values()), "H2": all(v["H2"] for v in outcomes.values())}
print(pd.DataFrame(outcomes).T.to_string())
print("overall:", overall)

             H1    H2 gap_hist500 gap_ewma_t4
WTI        True  True    0.043135    0.002752
FX basket  True  True      0.0247    0.002618
overall: {'H1': True, 'H2': True}


In [9]:
# cost in calm periods
cost_rows = []
for name in assets:
    flag = crisis_flag(var_tables[name].dropna().index)
    table = var_tables[name].dropna()
    cost_rows.append({
        "asset": name,
        "calm_mean_var_ratio_ewma_t4_to_hist500": float(table.loc[~flag.values, "ewma_t4"].mean() / table.loc[~flag.values, "hist500"].mean()),
        "crisis_mean_var_ratio_ewma_t4_to_hist500": float(table.loc[flag.values, "ewma_t4"].mean() / table.loc[flag.values, "hist500"].mean()),
    })
cost_table = pd.DataFrame(cost_rows)
print(cost_table.round(3).to_string(index=False))

# clustering on full series
independence = {}
for name in assets:
    independence[name] = {m: christoffersen_independence(h.to_numpy())["p_value"] for m, h in hits[name].items()}
print()
print(pd.DataFrame(independence).round(4).to_string())

    asset  calm_mean_var_ratio_ewma_t4_to_hist500  crisis_mean_var_ratio_ewma_t4_to_hist500
      WTI                                   0.884                                     1.587
FX basket                                   0.982                                     1.395

                WTI  FX basket
hist250      0.0090     0.0010
hist500      0.0000     0.0032
hist1000     0.0000     0.0144
ewma_normal  0.1652     0.8283
ewma_t4      0.4508     0.3687


## 13. Visualization

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
periods = ["gfc", "oil_collapse", "covid", "ukraine", "crisis_pooled", "calm"]
models_plot = ["hist500", "ewma_t4"]
colors = {"hist500": COLORS["second"], "ewma_t4": COLORS["main"]}
for ax, name in zip(axes, assets):
    for j, model in enumerate(models_plot):
        sub = evaluation[(evaluation["asset"] == name) & (evaluation["model"] == model)].set_index("period").loc[periods]
        positions = np.arange(len(periods)) + (j - 0.5) * 0.38
        ax.bar(positions, sub["rate"] * 100, width=0.38, color=colors[model], label=model)
    ax.axhline(1.0, color="black", linewidth=1, linestyle="--")
    ax.set_xticks(np.arange(len(periods)))
    ax.set_xticklabels(periods, rotation=30, ha="right")
    ax.set_title(name)
axes[0].set_ylabel("exception rate (percent)")
axes[0].legend()
fig.suptitle("Exception rate by period, 99 percent VaR, dashed line is the expected 1 percent")
fig.tight_layout()
save_figure(fig, MODULE, "exception_rate_by_period.png")
plt.show()

In [11]:
fig, ax = plt.subplots(figsize=(10, 4.2))
r = assets["WTI"]
table = var_tables["WTI"].dropna()
zoom = slice("2007-06-01", "2010-06-30")
ax.plot(r.loc[table.index][zoom].index, -r.loc[table.index][zoom] * 100, color=COLORS["grey"], linewidth=0.6, label="daily loss")
ax.plot(table[zoom].index, table["hist500"][zoom] * 100, color=COLORS["second"], label="hist500 VaR")
ax.plot(table[zoom].index, table["ewma_t4"][zoom] * 100, color=COLORS["main"], label="ewma_t4 VaR")
start, end = windows["gfc"]
ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color="#f0e0a0", alpha=0.4, label="crisis window")
ax.set_ylabel("percent of price")
ax.set_title("WTI daily loss and 99 percent VaR around the 2008 crisis")
ax.legend(loc="upper left")
save_figure(fig, MODULE, "wti_gfc_var_paths.png")
plt.show()

## 14. Interpretation

In [12]:
for name in assets:
    sub = evaluation[(evaluation["asset"] == name) & (evaluation["period"].isin(["crisis_pooled", "calm"]))]
    h = sub[sub["model"] == "hist500"].set_index("period")
    e = sub[sub["model"] == "ewma_t4"].set_index("period")
    print(f"{name}: hist500 crisis rate {h.loc['crisis_pooled', 'rate'] * 100:.2f} percent "
          f"({int(h.loc['crisis_pooled', 'exceptions'])} of {int(h.loc['crisis_pooled', 'days'])} days), "
          f"calm rate {h.loc['calm', 'rate'] * 100:.2f} percent. "
          f"ewma_t4 crisis rate {e.loc['crisis_pooled', 'rate'] * 100:.2f} percent, calm rate {e.loc['calm', 'rate'] * 100:.2f} percent.")
print()
print("H1 supported on both assets:", overall["H1"])
print("H2 supported on both assets:", overall["H2"])

WTI: hist500 crisis rate 5.31 percent (50 of 941 days), calm rate 1.09 percent. ewma_t4 crisis rate 1.28 percent, calm rate 1.13 percent.
FX basket: hist500 crisis rate 3.47 percent (33 of 951 days), calm rate 0.88 percent. ewma_t4 crisis rate 1.26 percent, calm rate 0.69 percent.

H1 supported on both assets: True
H2 supported on both assets: True


**How to read it.** A 99 percent model should be wrong on 1 day in 100 in every kind of period. When the historical model is
wrong several times more often in a crisis than in calm years, its VaR is a measure of the **past**, not of current risk. The responsive
model repairs most of that gap in a crisis, and the cost sits on the other side of the table: its VaR is higher or lower than the historical
VaR in calm times by the ratio printed in the cost table. Whether that cost is acceptable is a **business decision**, not a statistical one.

Clusters matter too. Exceptions in a crisis come in bunches, so the exact intervals above are narrower than the true uncertainty, and the
Kupiec test over pooled days is more likely to reject than its nominal 5 percent. The Christoffersen p values by model show how much
clustering remains.

## 15. Experiments

In [13]:
def pooled_rates(r, series):
    common = series.dropna().index
    flag = crisis_flag(common)
    h = exception_series(r.loc[common], series.loc[common])
    return float(h[flag].mean()), float(h[~flag].mean()), int(len(common))


grid_rows = []
for name, r in assets.items():
    x = r.to_numpy()
    configs = {}
    for window in [250, 500, 1000]:
        configs[f"hist{window}"] = pd.Series(rolling_historical_var(x, window, ALPHA), index=r.index)
    for decay in [0.94, 0.97]:
        for nu in [None, 3.5, 4, 6, 10]:
            label = f"ewma{decay}_nu{nu}"
            configs[label] = pd.Series(ewma_var(x, decay, ALPHA, nu=nu), index=r.index)
    start_day = max(s.dropna().index.min() for s in configs.values())
    for label, series in configs.items():
        series = series[series.index >= start_day]
        crisis_rate, calm_rate, days = pooled_rates(r, series)
        grid_rows.append({"asset": name, "config": label, "crisis_rate": crisis_rate, "calm_rate": calm_rate, "days": days})
grid = pd.DataFrame(grid_rows)
grid["crisis_gap"] = (grid["crisis_rate"] - EXPECTED_RATE).abs()
print(grid.pivot(index="config", columns="asset", values="crisis_rate").round(4).to_string())
print()
print(grid.pivot(index="config", columns="asset", values="calm_rate").round(4).to_string())

asset            FX basket     WTI
config                            
ewma0.94_nu10       0.0147  0.0181
ewma0.94_nu3.5      0.0126  0.0128
ewma0.94_nu4        0.0126  0.0128
ewma0.94_nu6        0.0137  0.0149
ewma0.94_nuNone     0.0189  0.0234
ewma0.97_nu10       0.0158  0.0159
ewma0.97_nu3.5      0.0137  0.0128
ewma0.97_nu4        0.0137  0.0128
ewma0.97_nu6        0.0137  0.0149
ewma0.97_nuNone     0.0189  0.0213
hist1000            0.0358  0.0574
hist250             0.0294  0.0351
hist500             0.0347  0.0531

asset            FX basket     WTI
config                            
ewma0.94_nu10       0.0096  0.0155
ewma0.94_nu3.5      0.0069  0.0110
ewma0.94_nu4        0.0069  0.0113
ewma0.94_nu6        0.0081  0.0134
ewma0.94_nuNone     0.0138  0.0191
ewma0.97_nu10       0.0081  0.0129
ewma0.97_nu3.5      0.0052  0.0093
ewma0.97_nu4        0.0052  0.0093
ewma0.97_nu6        0.0073  0.0107
ewma0.97_nuNone     0.0119  0.0176
hist1000            0.0079  0.0092
hist250            

In [14]:
# does h2 survive
baseline = grid[grid["config"] == "hist500"].set_index("asset")["crisis_gap"]
ewma_rows = grid[grid["config"].str.startswith("ewma") & ~grid["config"].str.endswith("None")]
closer = ewma_rows.assign(closer=ewma_rows.apply(lambda row: row["crisis_gap"] < baseline[row["asset"]], axis=1))
share_closer = float(closer["closer"].mean())
print("t based ewma configurations that beat hist500 on crisis gap:", closer["closer"].sum(), "of", len(closer))
print("share:", round(share_closer, 3))

t based ewma configurations that beat hist500 on crisis gap: 16 of 16
share: 1.0


This grid tests whether H2 depends on the one setting fixed in advance (decay 0.94 and 4 degrees of freedom). If most settings agree, the
conclusion is not an artefact of that choice. Note that the evaluation days here start at the first day where every configuration exists, so the sample can differ slightly from the main table.

## 16. Results and Comparisons

In [15]:
criteria = {
    "scratch_matches_library_no_lookahead_1e-12": bool(worst_gap < 1e-12),
    "windows_inside_data_with_80_days": bool(all(
        len(hits[name]["hist500"][start:end]) >= 80 for name in assets for start, end in windows.values()
    )),
    "excluded_wti_days_match_independent_rule": bool(set(excluded_days) == expected_excluded),
    "hypotheses_evaluated_for_both_assets": bool(all(set(v) >= {"H1", "H2"} for v in outcomes.values())),
}
print(criteria)

metrics = {
    "assets": {name: {"returns": int(len(r)), "evaluation_days": int(len(evaluation_days[name]))} for name, r in assets.items()},
    "excluded_wti_return_days": [d.date().isoformat() for d in excluded_days],
    "crisis_windows": windows,
    "hypothesis_outcomes_by_asset": outcomes,
    "hypothesis_outcomes_overall": overall,
    "evaluation_table": evaluation.to_dict(orient="records"),
    "cost_in_calm_and_crisis": cost_table.to_dict(orient="records"),
    "christoffersen_independence_p": independence,
    "sensitivity_grid": grid.to_dict(orient="records"),
    "share_t_ewma_configs_beating_hist500": share_closer,
    "scratch_vs_library_max_gap": float(worst_gap),
    "acceptance_criteria": criteria,
}
config = {"alpha": ALPHA, "windows": windows, "currencies": currencies, "hist_windows": [250, 500, 1000],
          "ewma_decay": 0.94, "ewma_nu": 4, "seed": SEED}
save_config(MODULE, config)
path = save_result(
    MODULE,
    methodology="Pre stated hypotheses on exception rates of a historical and an EWMA t VaR in crisis windows, Kupiec tests, exact intervals, sensitivity grid",
    data_kind="empirical",
    dataset="datasets/raw/wti_spot_daily.csv and datasets/processed/fx_daily_2000.csv",
    n_obs=int(sum(len(r) for r in assets.values())),
    seed=SEED,
    parameters=config,
    assumptions=[
        "crisis windows are calendar dates chosen by the author from well known events",
        "equal dollar weight FX basket rebalanced daily",
        "days with undefined WTI returns are dropped",
        "the author ran an exploratory version before fixing the rules",
    ],
    metrics=metrics,
    baseline="500 day historical VaR at 99 percent",
    limitations=[
        "two assets only, one FX basket and one commodity",
        "crisis windows are few and chosen with hindsight about which periods were crises",
        "exceptions cluster, so exact intervals and Kupiec p values are too optimistic",
        "single day VaR on linear positions, no ten day horizon and no capital calculation",
    ],
    figures=["exception_rate_by_period.png", "wti_gfc_var_paths.png"],
)
print("saved", path.name)

{'scratch_matches_library_no_lookahead_1e-12': True, 'windows_inside_data_with_80_days': True, 'excluded_wti_days_match_independent_rule': True, 'hypotheses_evaluated_for_both_assets': True}
saved var_crisis_research.json


## 17. Limitations and Model Risk

1. **Few events.** Four crisis windows are four draws from history. The pooled test treats their days as independent, which they are not.
2. **Hindsight in the windows.** The author knew which periods were crises. The dates were fixed from events, not from VaR failures, but the choice is still not blind.
3. **Exploration before rules.** An exploratory run came before the written rules. The rules were not changed after it, but the author cannot prove a blind test.
4. **Two assets.** An oil price and a currency basket say little about a loan book or an interest rate book.
5. **The worst day is missing.** The negative oil price day is dropped because a return is undefined. A real desk must handle that day, and a model that cannot represent it has a **model risk** that no backtest on the remaining days will show.
6. **Single day, linear positions.** No ten day horizon, no options, no liquidity horizon.
7. **Fixed settings.** Decay 0.94 and 4 degrees of freedom were set in advance, not estimated. The sensitivity grid tests part of this.
8. **Procyclicality is not solved.** A responsive VaR is higher in crises. If it drives capital or limits, it can force selling at the worst time. This notebook does not measure that effect.

## 18. Practical Banking Application

**Who does this work.** Market risk analysts in the second line of defence run backtests and write the model performance note. Model validation
repeats them independently. The committee decides.

**A usable note for the committee.**

1. State the model and the period tested.
2. State the exception rate in calm and in stress, with the count of days, as in the table above.
3. State what the responsive alternative would have done, including its cost in calm periods.
4. State the limits (above) in two lines.
5. Ask for a decision: keep, add an overlay, or move to the responsive model, with a review date.

**Typical actions when a model fails in stress.** Add a stressed period to the calibration window. Run both models and use the larger VaR. Lower the limit
usage trigger. Document the decision and the owner. None of these is recommended here, because the evidence in this notebook covers two assets only.

## 19. Exercises

1. Add a third model, a 250 day window with an exponentially weighted historical simulation, and test H2 for it.
2. Replace the calendar windows by a rule based on realised volatility (for example the top decile of 20 day volatility) and repeat. Does H1 survive?
3. Use blocks of 20 days to bootstrap the pooled crisis exception rate and compare the interval with the exact one.
4. Add the 2020 negative oil price day by using price differences instead of returns for that day. How large is the loss compared with the VaR?
5. Run the test on a currency pair that is not in the basket. What changes?

## 20. References

Entries are grouped by type. A reference marked NOT VERIFIED must be checked before it is cited.

- **Kupiec, P. H. (1995).** Techniques for verifying the accuracy of risk measurement models. Finance and Economics Discussion Series 95-24, Board of Governors of the Federal Reserve System. https://fraser.stlouisfed.org/title/1491/item/717921
  - Type: academic. Status: current. Check: verified: Fed FRASER and EconPapers records, web search 2026-11-25
- **Christoffersen, P. F. (1998).** Evaluating interval forecasts. International Economic Review 39(4):841-862. https://doi.org/10.2307/2527341
  - Type: academic. Status: current. Check: verified: RePEc IDEAS and EconPapers records, web search 2026-11-25
- **Basel Committee on Banking Supervision (2019).** Minimum capital requirements for market risk (BCBS d457). Bank for International Settlements, 14 January 2019 (rev. February 2019), consolidated into the Basel Framework. https://www.bis.org/bcbs/publ/d457.htm
  - Type: regulatory. Status: integrated into the consolidated Basel Framework. Check: verified: BIS page, web search 2026-11-25